# Train Binary Prostate Lesion Segmentation (5-Fold Cross-Validation)

This notebook runs 5-fold cross-validation training for **binary lesion segmentation** using RRUNet3D with multi-parametric inputs.

**Segmentation Classes:**
- 0 = Background
- 1 = Lesion

**Input Modalities:**
- T2-weighted (T2) - required
- ADC (Apparent Diffusion Coefficient) - optional, will use zeros if missing
- High b-value (HIGHB) - optional, will use zeros if missing

**Key Features:**
- Multi-parametric input (3 channels: T2 + ADC + HIGHB)
- ROI cropping based on organ mask with 32-voxel margin
- 0.5mm isotropic spacing (matches inference)
- Per-channel z-score normalization
- 5-fold ensemble for robust predictions
- **Handles missing lesion masks**: If a lesion mask doesn't exist (no tumor detected), an empty mask (all zeros) will be automatically created

**Prerequisites:**
- Ensure `data/preprocessed/t2/` and `data/preprocessed/organ_masks/` contain matching files.
- Organ masks are required for ROI cropping (from organ segmentation training).
- **Lesion masks are optional**: Missing lesion masks (no tumor detected) will be handled automatically by creating empty masks.
- If present, lesion masks should be binary: 0=background, 1=lesion
- ADC and HIGHB are optional but recommended for better performance.
- Fold-specific CSV files will be created automatically from `data/train.csv` if needed.
- Adjust `configs/lesion.yaml` if needed (should have `in_channels: 3`, `out_channels: 2`).


In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
repo_root = os.path.abspath("..") if os.getcwd().endswith("notebooks") else os.path.abspath(".")
if os.getcwd().endswith("notebooks"):
    os.chdir(repo_root)
print("CWD:", os.getcwd())


CWD: /home/anson/work/research-contributions/prostate-mri-lesion-seg


In [2]:
# Setup: Create fold-specific CSV files if needed
from pathlib import Path
import pandas as pd

splits_dir = Path("annotations/splits")
splits_dir.mkdir(parents=True, exist_ok=True)

# Check if CSV files exist
missing_folds = []
for fold in range(5):
    if not (splits_dir / f"fold{fold}_train.csv").exists():
        missing_folds.append(fold)

if missing_folds:
    print(f"Creating fold-specific CSV files for folds: {missing_folds}")
    df = pd.read_csv("data/train.csv")
    train_df = df[df['fold'] >= 0].copy()
    
    print(f"Total training patients: {len(train_df)}")
    print(f"Fold distribution: {train_df['fold'].value_counts().sort_index().to_dict()}")
    
    for fold in range(5):
        val_mask = train_df['fold'] == fold
        val_ids = train_df[val_mask]['ID'].tolist()
        train_mask = train_df['fold'] != fold
        train_ids = train_df[train_mask]['ID'].tolist()
        
        print(f"Fold {fold}: Train={len(train_ids)}, Val={len(val_ids)}")
        
        train_csv_df = pd.DataFrame({'subject_id': train_ids})
        train_csv_df.to_csv(splits_dir / f"fold{fold}_train.csv", index=False)
        
        val_csv_df = pd.DataFrame({'subject_id': val_ids})
        val_csv_df.to_csv(splits_dir / f"fold{fold}_val.csv", index=False)
    
    print(f"\n✓ Created fold-specific CSV files in {splits_dir}")
else:
    print("✓ Fold-specific CSV files already exist")


✓ Fold-specific CSV files already exist


In [3]:
# 5-Fold Cross-Validation Training for Lesion Segmentation
from training.engine import train_lesion_from_config
from training.utils import load_yaml
from pathlib import Path
import logging
import copy
import yaml
import tempfile
import os

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

config_path = "configs/lesion.yaml"
splits_dir = Path("annotations/splits")

# Load base config
cfg = load_yaml(config_path)
print(f"Model: {cfg['model']['type']} with {cfg['model']['in_channels']} input channels (T2+ADC+HIGHB)")
print(f"Output: {cfg['model']['out_channels']} channels (background=0, lesion=1)")
print(f"Spacing: {cfg['preprocess']['spacing']} mm")
print(f"ROI margin: {cfg['preprocess']['roi_margin']} voxels")
print(f"Training all 5 folds...\n")

# Train all folds
all_artifacts = []
for fold in range(5):
    print(f"\n{'='*60}")
    print(f"Starting Fold {fold} Training")
    print(f"{'='*60}")
    
    # Create a deep copy of config for this fold
    fold_cfg = copy.deepcopy(cfg)
    
    # Update config to use fold-specific CSV files
    train_csv = splits_dir / f"fold{fold}_train.csv"
    val_csv = splits_dir / f"fold{fold}_val.csv"
    
    fold_cfg["dataset"]["train_split_csv"] = str(train_csv)
    fold_cfg["dataset"]["val_split_csv"] = str(val_csv)
    fold_cfg["dataset"]["scan_all"] = False  # Use CSV files
    
    # Update output directory to include fold number
    base_exp_dir = fold_cfg["output"]["exp_dir"]
    fold_cfg["output"]["exp_dir"] = str(Path(base_exp_dir) / f"fold{fold}")
    
    print(f"Train CSV: {train_csv}")
    print(f"Val CSV: {val_csv}")
    print(f"Output directory: {fold_cfg['output']['exp_dir']}\n")
    
    # Create temporary YAML file for this fold (since train_lesion_from_config expects a file path)
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as tmp_file:
        yaml.dump(fold_cfg, tmp_file, default_flow_style=False)
        tmp_config_path = tmp_file.name
    
    try:
        # Train using the temporary config file
        artifacts = train_lesion_from_config(tmp_config_path)
        all_artifacts.append((fold, artifacts))
        print(f"\n✓ Fold {fold} training completed!")
        print(f"  Best checkpoint: {artifacts.best_ckpt}")
        print(f"  Metrics CSV: {artifacts.metrics_csv}")
    except Exception as e:
        print(f"\n✗ Fold {fold} training failed: {e}")
        import traceback
        traceback.print_exc()
        continue
    finally:
        # Clean up temporary config file
        if os.path.exists(tmp_config_path):
            os.unlink(tmp_config_path)

# Summary
print(f"\n{'='*60}")
print("5-Fold Cross-Validation Training Summary")
print(f"{'='*60}")
print(f"Successfully trained: {len(all_artifacts)}/5 folds")
for fold, artifacts in all_artifacts:
    print(f"  Fold {fold}: {artifacts.best_ckpt}")

# Store artifacts for later use
fold_artifacts = {fold: artifacts for fold, artifacts in all_artifacts}


Model: rrunet3d with 3 input channels (T2+ADC+HIGHB)
Output: 2 channels (background=0, lesion=1)
Spacing: [0.5, 0.5, 0.5] mm
ROI margin: 32 voxels
Training all 5 folds...


Starting Fold 0 Training
Train CSV: annotations/splits/fold0_train.csv
Val CSV: annotations/splits/fold0_val.csv
Output directory: experiments/lesion/fold0

Using device: cuda


Loading dataset: 100%|██████████| 26/26 [00:24<00:00,  1.07it/s]


Epoch 1/60


`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


  step 10/130 - loss: 1.0539
  step 20/130 - loss: 1.0391
  step 30/130 - loss: 1.0366
  step 40/130 - loss: 1.0283
  step 50/130 - loss: 1.0217
  step 60/130 - loss: 1.0220
  step 70/130 - loss: 1.0226
  step 80/130 - loss: 1.0220
  step 90/130 - loss: 1.0208
  step 100/130 - loss: 1.0213
  step 110/130 - loss: 1.0206
  step 120/130 - loss: 1.0205
  step 130/130 - loss: 1.0195
  val mean dice: 0.0809
Epoch 2/60
  step 10/130 - loss: 0.9971
  step 20/130 - loss: 1.0048
  step 30/130 - loss: 1.0001
  step 40/130 - loss: 1.0008
  step 50/130 - loss: 1.0010
  step 60/130 - loss: 1.0011
  step 70/130 - loss: 0.9985
  step 80/130 - loss: 0.9992
  step 90/130 - loss: 1.0010
  step 100/130 - loss: 1.0020
  step 110/130 - loss: 1.0027
  step 120/130 - loss: 1.0029
  step 130/130 - loss: 1.0031
  val mean dice: 0.1407
Epoch 3/60
  step 10/130 - loss: 0.9721
  step 20/130 - loss: 0.9849
  step 30/130 - loss: 0.9868
  step 40/130 - loss: 0.9884
  step 50/130 - loss: 0.9860
  step 60/130 - loss: 0

...

  step 10/130 - loss: 1.0527
  step 20/130 - loss: 1.0405
  step 30/130 - loss: 1.0358
  step 40/130 - loss: 1.0291
  step 50/130 - loss: 1.0288
  step 60/130 - loss: 1.0283
  step 70/130 - loss: 1.0287
  step 80/130 - loss: 1.0268
  step 90/130 - loss: 1.0257
  step 100/130 - loss: 1.0261
  step 110/130 - loss: 1.0248
  step 120/130 - loss: 1.0222
  step 130/130 - loss: 1.0220
  val mean dice: 0.1262
Epoch 2/60
  step 10/130 - loss: 0.9961
  step 20/130 - loss: 1.0048
  step 30/130 - loss: 1.0096
  step 40/130 - loss: 1.0097
  step 50/130 - loss: 1.0122
  step 60/130 - loss: 1.0087
  step 70/130 - loss: 1.0100
  step 80/130 - loss: 1.0072
  step 90/130 - loss: 1.0056
  step 100/130 - loss: 1.0050
  step 110/130 - loss: 1.0048
  step 120/130 - loss: 1.0030
  step 130/130 - loss: 1.0025
  val mean dice: 0.2102
Epoch 3/60
  step 10/130 - loss: 1.0106
  step 20/130 - loss: 0.9963
  step 30/130 - loss: 0.9982
  step 40/130 - loss: 0.9975
  step 50/130 - loss: 0.9954
  step 60/130 - loss: 0

Loading dataset: 100%|██████████| 26/26 [00:29<00:00,  1.13s/it]

Epoch 1/60


  step 10/131 - loss: 1.0556
  step 20/131 - loss: 1.0447
  step 30/131 - loss: 1.0361
  step 40/131 - loss: 1.0307
  step 50/131 - loss: 1.0312
  step 60/131 - loss: 1.0291
  step 70/131 - loss: 1.0262
  step 80/131 - loss: 1.0255
  step 90/131 - loss: 1.0231
  step 100/131 - loss: 1.0222
  step 110/131 - loss: 1.0189
  step 120/131 - loss: 1.0175
  step 130/131 - loss: 1.0164
  val mean dice: 0.0931
Epoch 2/60
  step 10/131 - loss: 1.0113
  step 20/131 - loss: 1.0103
  step 30/131 - loss: 1.0116
  step 40/131 - loss: 1.0078
  step 50/131 - loss: 1.0072
  step 60/131 - loss: 1.0063
  step 70/131 - loss: 1.0046
  step 80/131 - loss: 1.0052
  step 90/131 - loss: 1.0053
  step 100/131 - loss: 1.0051
  step 110/131 - loss: 1.0035
  step 120/131 - loss: 1.0033
  step 130/131 - loss: 1.0038
  val mean dice: 0.2823
Epoch 3/60
  step 10/131 - loss: 1.0178
  step 20/131 - loss: 1.0082
  step 30/131 - loss: 1.0073
  step 40/131 - loss: 1.0042
  step 50/131 - loss: 1.0036
  step 60/131 - loss: 1

KeyboardInterrupt: 